In [1]:
import numpy as np
import scipy as sc
import matplotlib.pyplot as plt
import pandas as pd
import ast

from tqdm import tqdm
from numpy.polynomial.legendre import legvander

plt.rcParams.update({'axes.xmargin': 0.0,
                     'axes.ymargin': 0.0
                    }
                   )

In [2]:
#---------------------------
# reading in edge timeseries
#---------------------------

ts = pd.read_csv('16-11-2026_BD4-edge.csv')
ts

,Timestamp,Value
0,0.00005,"{{1.4603275644028337, 18.902315292395038}, {1...."
1,0.00015,"{{1.4284724689729438, 18.755675538314808}, {1...."
2,0.00025,"{{1.410914759664394, 18.805054328142806}, {1.3..."
3,0.00035,"{{1.3945837657834872, 18.68816487760446}, {1.3..."
4,0.00045,"{{1.3965805403519524, 18.687789263981802}, {1...."
...,...,...
4795,0.47955,"{{1.8400533035830589, 24.17888615642901}, {1.7..."
4796,0.47965,"{{1.6672701019774834, 23.220400583456534}, {1...."
4797,0.47975,"{{1.6285087978561865, 23.824942278853236}, {1...."
4798,0.47985,"{{1.8089686861597296, 24.257046480617532}, {1...."


In [3]:
#-------------------------------------------
# changing curly brackets to square brackets
#-------------------------------------------

values = []

for val in tqdm(range(len(ts))):

    # getting coordinate list at each index
    coords = ts["Value"][val]

    # Replacing curly brackets with square brackets
    coords = ast.literal_eval(coords.replace("{","[").replace("}","]"))

    values.append(coords)

# changing edge data to string
ts["Value"] = values

100%|█████████████████████████████████████████████████| 4800/4800 [00:08<00:00, 565.21it/s]


In [4]:
#----------------------------------------
# Extracting fourier amplitudes up to n=4
#----------------------------------------

data = ts

def calculate_modes(data, max_n=4):

    times = []
    coefficients = []

    for _, frame in data.iterrows():

        # getting timeseries information
        t = frame["Timestamp"]
        coordinates = np.array(frame["Value"])

        theta = coordinates[:, 0]
        r = coordinates[:, 1]

        # constructing Legendre polynomial matrix
        P = legvander(np.cos(theta), max_n)

        # least-squares solution
        C, residuals, rank, s = np.linalg.lstsq(P, r, rcond=None)

        times.append(t)
        coefficients.append(C)

    return np.array(times), np.array(coefficients)

times, modes = calculate_modes(ts, max_n=8)

In [5]:
C4 = np.abs(modes[:,4])

In [123]:
#---------------------
# Saving data into CSV 
#---------------------
fourierAmplitude = pd.DataFrame({'Timestamp': times, '|C4|': C4})
fourierAmplitude.to_csv('|C4|.csv', index=False)